# Outliers from Scratch: Boxplot, IQR & Z-Score

**Goal:** Detect unusual values in a simple numeric column using three beginner tools:
1. **Boxplot** (see it)
2. **IQR fences** (decide it)
3. **Z-score** (measure distance from the mean)

**Tiny example:** customer ages with one suspicious value (`95`).


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Tiny classroom dataset — most ages are ~22–32, one extreme value
ages = [22, 24, 25, 26, 27, 28, 29, 30, 31, 32, 95]
df = pd.DataFrame({"age": ages})
df


## 1. Look at the data first

Before any formula: sort, describe, and plot a histogram. Does anything look weird?


In [ ]:
print("Sorted ages:", sorted(df["age"].tolist()))
display(df["age"].describe())

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.hist(df["age"], bins=[20, 30, 40, 50, 60, 70, 80, 90, 100], edgecolor="white")
ax.set_xlabel("Age")
ax.set_ylabel("Count")
ax.set_title("Age distribution — notice the lonely bar near 90–100")
plt.tight_layout()
plt.show()


## 2. Boxplot = a picture of Q1, median, Q3

- **Box** = middle 50% of the data (Q1 → Q3)
- **Line inside the box** = median
- **Whiskers** = farthest non-outlier points
- **Dots outside** = outliers


In [ ]:
fig, ax = plt.subplots(figsize=(8, 2.8))
ax.boxplot(df["age"], vert=False, widths=0.5)
ax.set_xlabel("Age")
ax.set_title("Boxplot of ages (matplotlib marks outliers as circles)")
ax.set_yticks([])
plt.tight_layout()
plt.show()


## 3. IQR method (best beginner default)

Steps:
1. `Q1` = 25th percentile, `Q3` = 75th percentile
2. `IQR = Q3 − Q1`
3. Low fence = `Q1 − 1.5 × IQR`
4. High fence = `Q3 + 1.5 × IQR`
5. Flag any value **outside** the fences

Why beginners like IQR: it does **not** assume a bell curve, and one extreme value barely moves Q1/Q3.


In [ ]:
q1 = df["age"].quantile(0.25)
q3 = df["age"].quantile(0.75)
iqr = q3 - q1
low_fence = q1 - 1.5 * iqr
high_fence = q3 + 1.5 * iqr

print(f"Q1 = {q1:.2f}")
print(f"Q3 = {q3:.2f}")
print(f"IQR = {iqr:.2f}")
print(f"Low fence  = Q1 - 1.5*IQR = {low_fence:.2f}")
print(f"High fence = Q3 + 1.5*IQR = {high_fence:.2f}")

iqr_mask = (df["age"] < low_fence) | (df["age"] > high_fence)
iqr_outliers = df.loc[iqr_mask, "age"]
print("\nIQR outliers:", iqr_outliers.tolist())
df.assign(iqr_outlier=iqr_mask)


In [ ]:
# Visual: number line with IQR fences
fig, ax = plt.subplots(figsize=(9, 2.6))
ax.scatter(df.loc[~iqr_mask, "age"], np.zeros( (~iqr_mask).sum()), s=60, label="normal", zorder=3)
ax.scatter(df.loc[iqr_mask, "age"], np.zeros(iqr_mask.sum()), s=90, facecolors="none",
           edgecolors="C1", linewidths=2, label="outlier", zorder=4)
ax.axvline(low_fence, color="gray", linestyle="--", label=f"low fence {low_fence:.1f}")
ax.axvline(high_fence, color="gray", linestyle="--", label=f"high fence {high_fence:.1f}")
ax.axvspan(q1, q3, alpha=0.2, color="C0", label="IQR box (Q1–Q3)")
ax.set_yticks([])
ax.set_xlabel("Age")
ax.set_title("IQR fences on a number line")
ax.legend(loc="upper center", ncol=2, fontsize=9)
plt.tight_layout()
plt.show()


## 4. Z-score method

Formula:

`z = (x − mean) / std`

Common rule: `|z| > 3` → outlier (or `|z| > 2` for a stricter screen).

Caution: mean and std are **pulled toward** outliers, so prefer IQR when data is skewed.


In [ ]:
mu = df["age"].mean()
sigma = df["age"].std(ddof=0)  # population std for teaching clarity
df_z = df.copy()
df_z["z"] = (df_z["age"] - mu) / sigma
df_z["abs_z"] = df_z["z"].abs()

Z_THRESHOLD = 3  # try changing to 2
df_z["z_outlier"] = df_z["abs_z"] > Z_THRESHOLD

print(f"mean = {mu:.2f}")
print(f"std  = {sigma:.2f}")
print(f"rule: |z| > {Z_THRESHOLD}")
print("Z-score outliers:", df_z.loc[df_z["z_outlier"], "age"].tolist())
df_z.round(2)


In [ ]:
# Visual: z-score number line with ± threshold bands
fig, ax = plt.subplots(figsize=(9, 2.8))
ax.axvline(mu, color="black", linewidth=1.5, label=f"mean {mu:.1f}")
ax.axvline(mu - Z_THRESHOLD * sigma, color="C1", linestyle="--", label=f"z=-{Z_THRESHOLD}")
ax.axvline(mu + Z_THRESHOLD * sigma, color="C1", linestyle="--", label=f"z=+{Z_THRESHOLD}")
ax.scatter(df_z.loc[~df_z["z_outlier"], "age"], np.zeros((~df_z["z_outlier"]).sum()),
           s=60, label="normal", zorder=3)
ax.scatter(df_z.loc[df_z["z_outlier"], "age"], np.zeros(df_z["z_outlier"].sum()),
           s=90, facecolors="none", edgecolors="C1", linewidths=2, label="outlier", zorder=4)
for _, row in df_z.loc[df_z["z_outlier"]].iterrows():
    ax.text(row["age"], 0.08, f"{int(row['age'])}\nz={row['z']:.1f}",
            ha="center", va="bottom", color="C1", fontsize=9)
ax.set_yticks([])
ax.set_xlabel("Age")
ax.set_title("Z-score fences on a number line")
ax.legend(loc="upper center", ncol=3, fontsize=9)
plt.tight_layout()
plt.show()


## 5. Compare IQR vs Z-score on this example


In [ ]:
compare = pd.DataFrame({
    "age": df["age"],
    "iqr_outlier": iqr_mask,
    "z_outlier": df_z["z_outlier"],
    "z_score": df_z["z"].round(2),
})
compare


## 6. What to do after you find an outlier?

1. **Investigate** — is `95` a real age or a typo (meant `35`)?
2. **Remove / quarantine** — drop bad rows and log why (like Day 6 validation)
3. **Cap (winsorize)** — clamp extreme values to the fence so they stop dominating the model


In [ ]:
# Example A: quarantine (keep a clean copy + a bad-row log)
clean = df.loc[~iqr_mask].copy()
quarantine = df.loc[iqr_mask].copy()
quarantine["failure_reason"] = "age outside 1.5*IQR fences"
print("Clean rows:", len(clean), "| Quarantined:", len(quarantine))
display(quarantine)

# Example B: winsorize / cap to the high fence
capped = df.copy()
capped["age_capped"] = capped["age"].clip(lower=low_fence, upper=high_fence)
capped


## Memory hook

| Tool | Job |
| --- | --- |
| **Boxplot** | Shows the unusual points |
| **IQR** | Decides with robust fences (`1.5 × IQR`) |
| **Z-score** | Measures distance in std units (`|z| > 3`) |

**Beginner default:** start with boxplot + IQR. Use z-score after checking the distribution is roughly normal.


In [ ]:
# Self-check (all should print True)
assert 95 in iqr_outliers.tolist()
assert df_z.loc[df_z["age"] == 95, "z_outlier"].iloc[0] == True
assert clean["age"].max() < 95
assert capped["age_capped"].max() <= high_fence + 1e-9
print("All self-checks passed.")
